# Blue Lobster — Data Wrangler
Extracts daily sea surface temperature (SST) from NOAA OISST NetCDF files for Atlantic Canadian sub-regions and exports a clean CSV for the habitat suitability model.

**Input:** `Data/*.nc` — OISST AVHRR v02r01 daily files (0.25° global grid)  
**Output:** `Data/sst_regions.csv`  
**CSV consumer:** John's suitability scorer (`scorer.py` → `suitability.json`)


In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

DATA_DIR = Path("Data")
OUT_CSV  = DATA_DIR / "sst_regions.csv"

nc_files = sorted(DATA_DIR.glob("oisst-avhrr-v02r01.*.nc"))
print(f"Found {len(nc_files)} file(s):")
for f in nc_files:
    print(" ", f.name)


## Sub-region definitions

Bounding boxes roughly follow DFO Lobster Fishing Area (LFA) groupings and the Gulf of Maine baseline zone.  
`lon` values are in conventional −180/180 degrees.

| Region | Context |
|---|---|
| Gulf of Maine | Warming source zone; traditional US/Canadian harvest area |
| Bay of Fundy | Transitional corridor |
| SW Scotian Shelf | LFA 33–34; currently productive Canadian zone |
| NE Scotian Shelf | LFA 27–32 |
| Cape Breton | LFA 26 / 26A |
| Gulf of St. Lawrence | LFA 22–25; interior sea with distinct thermal regime |
| S Newfoundland | Emerging habitat as larvae survive further north |


In [ ]:
# (lat_min, lat_max), (lon_min, lon_max) — all in degrees N / degrees E (−180 to 180)
REGIONS = {
    "Gulf of Maine":        {"lat": (40.0, 44.0), "lon": (-71.0, -63.0)},
    "Bay of Fundy":         {"lat": (44.0, 46.5), "lon": (-68.0, -63.5)},
    "SW Scotian Shelf":     {"lat": (43.0, 45.5), "lon": (-66.5, -62.0)},
    "NE Scotian Shelf":     {"lat": (44.5, 47.0), "lon": (-62.0, -58.0)},
    "Cape Breton":          {"lat": (45.5, 47.5), "lon": (-62.0, -59.0)},
    "Gulf of St. Lawrence": {"lat": (46.0, 50.0), "lon": (-66.0, -60.0)},
    "S Newfoundland":       {"lat": (46.5, 50.5), "lon": (-56.0, -52.0)},
}

for name, bbox in REGIONS.items():
    print(f"{name:<25} lat {bbox['lat'][0]}–{bbox['lat'][1]}  lon {bbox['lon'][0]}–{bbox['lon'][1]}")


## Load and process OISST files

OISST stores longitude as 0–360°E. We remap to −180/180 with `assign_coords` + `sortby` so our region bounding boxes work directly. The raw SST is stored as int16 with `scale_factor = 0.01`; xarray decodes it automatically when `mask_and_scale=True`.


In [ ]:
def extract_date(path: Path) -> str:
    """Parse YYYYMMDD from filename → ISO date string."""
    stem = path.stem  # e.g. oisst-avhrr-v02r01.20250101
    raw = stem.split(".")[-1]  # '20250101'
    return f"{raw[:4]}-{raw[4:6]}-{raw[6:]}"


def remap_lons(ds: xr.Dataset) -> xr.Dataset:
    """Shift longitude axis from 0–360 to −180/180 and re-sort."""
    lon_180 = (ds.lon.values + 180) % 360 - 180
    ds = ds.assign_coords(lon=lon_180)
    return ds.sortby("lon")


def nanmean_safe(arr: np.ndarray) -> float:
    """np.nanmean that returns NaN (not a warning) when all values are NaN."""
    vals = arr.ravel()
    mask = ~np.isnan(vals)
    return float(vals[mask].mean()) if mask.any() else np.nan


def process_file(path: Path) -> list[dict]:
    """Return one dict per region for a single daily OISST file."""
    date_str = extract_date(path)

    ds = xr.open_dataset(path, mask_and_scale=True)
    ds = remap_lons(ds)

    # Drop size-1 dimensions (time, zlev) → 2-D (lat, lon) arrays
    sst  = ds["sst"].squeeze(drop=True)   # °C, NaN where land/ice
    anom = ds["anom"].squeeze(drop=True)  # °C anomaly from climatology
    ice  = ds["ice"].squeeze(drop=True)   # % sea ice concentration

    rows = []
    for region, bbox in REGIONS.items():
        lat_min, lat_max = bbox["lat"]
        lon_min, lon_max = bbox["lon"]

        sst_box  = sst.sel( lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
        anom_box = anom.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
        ice_box  = ice.sel( lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

        sst_vals = sst_box.values.ravel()
        valid    = ~np.isnan(sst_vals)
        n_valid  = int(valid.sum())

        rows.append({
            "date":       date_str,
            "region":     region,
            "lat_min":    lat_min,
            "lat_max":    lat_max,
            "lon_min":    lon_min,
            "lon_max":    lon_max,
            "sst_mean":   round(nanmean_safe(sst_vals),           3) if n_valid else np.nan,
            "sst_min":    round(float(np.nanmin(sst_vals)),        3) if n_valid else np.nan,
            "sst_max":    round(float(np.nanmax(sst_vals)),        3) if n_valid else np.nan,
            "sst_std":    round(float(np.nanstd(sst_vals)),        3) if n_valid else np.nan,
            "anom_mean":  round(nanmean_safe(anom_box.values),     3),
            "ice_mean":   round(nanmean_safe(ice_box.values),      3),
            "n_valid":    n_valid,
        })

    ds.close()
    return rows


# --- Run across all files ---
all_rows = []
for f in nc_files:
    print(f"Processing {f.name} …", end=" ")
    rows = process_file(f)
    all_rows.extend(rows)
    print("done")

df = pd.DataFrame(all_rows)
print(f"\nTotal rows: {len(df)}  ({df['date'].nunique()} date(s) × {df['region'].nunique()} regions)")


## Inspect results and export CSV


In [ ]:
preview_cols = ["date", "region", "sst_mean", "sst_min", "sst_max", "anom_mean", "ice_mean", "n_valid"]
display_df = df[preview_cols].copy()
display_df["above_20C"] = df["sst_mean"] > 20.0   # stress threshold for lobster
print(display_df.to_string(index=False))


In [ ]:
df.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(df)} rows → {OUT_CSV}")
print("\nColumn schema:")
for col in df.columns:
    print(f"  {col:<15} {df[col].dtype}")
